In [ ]:
!pip install ogx_client

In [2]:
from ogx_client import OgxClient
import rich

In [3]:
# Configuration
OGX_CONNECTION_URL = "http://ogxserver-with-inline-emb-service.llama.svc.cluster.local:8321"

In [43]:
# Initialize OGX client
client = OgxClient(base_url=OGX_CONNECTION_URL)

In [29]:
# List available models
models = client.models.list()
rich.print(models)

ListModelsResponse(
    data=[
        Model(
            id='vllm-inference/llama-32-3b-instruct',
            created=1780586374,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'llm',
                'provider_id': 'vllm-inference',
                'provider_resource_id': 'llama-32-3b-instruct'
            },
            object='model'
        ),
        Model(
            id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
            created=1780586374,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'embedding',
                'provider_id': 'sentence-transformers',
                'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': 768
            },
            object='model'
        )
    ],
    object='list'
)

In [44]:
response = client.chat.completions.create(
    model="vllm-inference/llama-32-3b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about open source"},
    ],
)
print(response.choices[0].message.content)	

Free code, shared delight
Community's gentle hand
Freedom's open code


### Test Response API with model inferencing

In [30]:
response = client.with_options(timeout=600.0).responses.create(
    model="vllm-inference/llama-32-3b-instruct",
    input="What is the capital of India?"
)

In [31]:
rich.print(response.output_text)

The capital of India is New Delhi.

### Test Response API with model inferencing and Tool Runtime

In [8]:
response = client.with_options(timeout=600.0).responses.create(
    model="vllm-inference/llama-32-3b-instruct",
    input="What is 12 multiplied by 7?",
    tools=[{
        "type": "mcp",
        "server_label": "MathOperationsServer",
        "server_url": "http://math-mcp-server.llama.svc.cluster.local:9000/sse", # "http://host.containers.internal:9000/sse",
    }],
)

In [9]:
rich.print(response.output_text)

The result of 12 multiplied by 7 is 84.

In [10]:
rich.print(response)

ResponseObject(
    id='resp_2cd11d2f-c8d4-48a2-8d11-38c9e3c52d81',
    created_at=1780584994,
    model='vllm-inference/llama-32-3b-instruct',
    output=[
        OutputOpenAIResponseOutputMessageMcpListTools(
            id='mcp_list_6621ac39-18fc-48f6-8211-76787c6b162c',
            server_label='MathOperationsServer',
            tools=[
                OutputOpenAIResponseOutputMessageMcpListToolsTool(
                    input_schema={
                        'properties': {
                            'a': {'title': 'A', 'type': 'integer'},
                            'b': {'title': 'B', 'type': 'integer'}
                        },
                        'required': ['a', 'b'],
                        'title': 'addArguments',
                        'type': 'object'
                    },
                    name='add',
                    description='\n    Add two numbers together.\n    \n    Args:\n        a: First number\n       
b: Second number\n        \n    Returns:\n        Sum of a and b\n    '
                ),
                OutputOpenAIResponseOutputMessageMcpListToolsTool(
                    input_schema={
                        'properties': {
                            'a': {'title': 'A', 'type': 'integer'},
                            'b': {'title': 'B', 'type': 'integer'}
                        },
                        'required': ['a', 'b'],
                        'title': 'multiplyArguments',
                        'type': 'object'
                    },
                    name='multiply',
                    description='\n    Multiply two numbers.\n    \n    Args:\n        a: First number\n        b: 
Second number\n        \n    Returns:\n        Product of a and b\n    '
                )
            ],
            type='mcp_list_tools'
        ),
        OutputOpenAIResponseOutputMessageMcpCall(
            id='fc_65e68e22-6e6b-40a5-85da-bf7119cdc094',
            arguments='{"a": "12", "b": "7"}',
            name='multiply',
            server_label='MathOperationsServer',
            error=None,
            output='84',
            type='mcp_call'
        ),
        OutputOpenAIResponseMessageOutput(
            content=[
                OutputOpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenA
IResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(
                    text='The result of 12 multiplied by 7 is 84.',
                    annotations=[],
                    logprobs=[],
                    type='output_text'
                )
            ],
            role='assistant',
            id='msg_9b69a70e-973f-4744-8d0d-a29443122085',
            status='completed',
            type='message'
        )
    ],
    status='completed',
    store=True,
    background=False,
    completed_at=1780585008,
    error=None,
    frequency_penalty=0.0,
    incomplete_details=None,
    instructions=None,
    max_output_tokens=None,
    max_tool_calls=None,
    metadata=None,
    object='response',
    parallel_tool_calls=True,
    presence_penalty=0.0,
    previous_response_id=None,
    prompt=None,
    prompt_cache_key=None,
    reasoning=None,
    safety_identifier=None,
    service_tier='default',
    temperature=1.0,
    text=Text(
        format=TextFormat(description=None, name=None, schema_=None, strict=None, type='text'),
        verbosity=None
    ),
    tool_choice='auto',
    tools=[ToolOpenAIResponseToolMcp(server_label='MathOperationsServer', allowed_tools=None, type='mcp')],
    top_logprobs=0,
    top_p=1.0,
    truncation='disabled',
    usage=Usage(
        input_tokens=982,
        input_tokens_details=UsageInputTokensDetails(cached_tokens=0),
        output_tokens=36,
        output_tokens_details=UsageOutputTokensDetails(reasoning_tokens=0),
        total_tokens=1018
    )
)

### Verify inference, vector io, tool runtime and files
Upload documents, create a vector store, and ask questions. OGX handles chunking, embedding, and retrieval:

In [12]:
# Checking if any vector db present
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_95f051ff-fcce-4bb2-ad36-ccf821b32953',
            created_at=1780500897,
            file_counts=FileCounts(cancelled=0, completed=0, failed=1, in_progress=0, total=1),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1780500897,
            metadata={
                'provider_id': 'milvus',
                'provider_vector_store_id': 'vs_95f051ff-fcce-4bb2-ad36-ccf821b32953',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_95f051ff-fcce-4bb2-ad36-ccf821b32953',
    object='list',
    first_id='vs_95f051ff-fcce-4bb2-ad36-ccf821b32953'
)

In [14]:
# deleting existing vector store
rich.print(client.vector_stores.delete("vs_95f051ff-fcce-4bb2-ad36-ccf821b32953"))

VectorStoreDeleteResponse(
    id='vs_95f051ff-fcce-4bb2-ad36-ccf821b32953',
    deleted=True,
    object='vector_store.deleted'
)

In [15]:
# Checking if any vector db present
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id='', object='list', first_id='')

In [18]:
# Extract LLM and embedding model details
llm_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "llm"
)

# Using specifically sentence-transformers because customized the config to use this inline model
embedding_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "embedding" and m.custom_metadata.get("provider_id") == "sentence-transformers"
)

model_id = llm_model.id
embedding_model_id = embedding_model.id
embedding_dimension = embedding_model.custom_metadata["embedding_dimension"]

print(f"LLM Model: {model_id}")
print(f"Embedding Model: {embedding_model_id}")
print(f"Embedding Dimension: {embedding_dimension}")

LLM Model: vllm-inference/llama-32-3b-instruct
Embedding Model: sentence-transformers/nomic-ai/nomic-embed-text-v1.5
Embedding Dimension: 768


In [19]:
# Create vector store with Milvus-lite
vector_store = client.vector_stores.create(
    name="techmart_policy_store",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "milvus",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

Created vector store: vs_2c24ed2f-d1cb-49d3-b102-b2d70cff2b72


In [20]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_2c24ed2f-d1cb-49d3-b102-b2d70cff2b72',
            created_at=1780585602,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1780585602,
            metadata={
                'provider_id': 'milvus',
                'provider_vector_store_id': 'vs_2c24ed2f-d1cb-49d3-b102-b2d70cff2b72',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_2c24ed2f-d1cb-49d3-b102-b2d70cff2b72',
    object='list',
    first_id='vs_2c24ed2f-d1cb-49d3-b102-b2d70cff2b72'
)

In [21]:
# Policy file
POLICY_FILE = "data/return-policy.txt"

# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info.id}")

Uploaded file: file-5e7b357e56524dfba3571674e94b3546


In [22]:
# Add file to vector store with chunking strategy
vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file)

VectorStoreFile(
    id='file-5e7b357e56524dfba3571674e94b3546',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1780585912,
    status='completed',
    vector_store_id='vs_2c24ed2f-d1cb-49d3-b102-b2d70cff2b72',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [23]:
INSTRUCTIONS_PROMPT = """You are a helpful and professional customer service assistant.

WORKFLOW:
1. Analyze the customer's question carefully
2. Use available tools to gather all necessary information
3. After gathering information, provide a COMPLETE, well-structured answer

RESPONSE REQUIREMENTS:
- Be clear, accurate, and professional
- Include all relevant details from the tools
- Structure your answer logically
- Provide actionable next steps when applicable
- Never stop after calling tools - always synthesize the final answer

IMPORTANT: You MUST provide a final answer after using tools. Be helpful, accurate, and thorough."""

In [24]:
# Test 1: General return policy question
query = "What is the return window for electronics?"

response = client.with_options(timeout=600.0).responses.create(
    model=model_id,
    input=query,
    instructions=INSTRUCTIONS_PROMPT,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        }
    ],
)

print("\n" + "="*80)
print(f"QUESTION: {query}")
print("="*80)
print(f"\nANSWER:\n{response.output_text}")
print("\n" + "="*80)


QUESTION: What is the return window for electronics?

ANSWER:
The return window for electronics at TechMart is as follows:

* Standard items: 30 days from delivery
* Electronics: 15 days from delivery
* Opened software and personalized items: Not accepted for return
* Items must be in original condition with original packaging intact, including all accessories, manuals, and tags.
* Refunds are processed within 5-7 business days after receiving the



### Verification with OpenAI

In [ ]:
!pip install openai

In [41]:
from openai import OpenAI
client = OpenAI(base_url="http://ogxserver-with-inline-emb-service.llama.svc.cluster.local:8321/v1", api_key="fake")

In [40]:
# List available models
models = client.models.list()
for m in models.data:
    print(f"  {m.id} ({m.object})")

  vllm-inference/llama-32-3b-instruct (model)
  sentence-transformers/nomic-ai/nomic-embed-text-v1.5 (model)


In [42]:
# Simple chat
response = client.chat.completions.create(
    model="vllm-inference/llama-32-3b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about open source"},
    ],
)
print(response.choices[0].message.content)

Code shared freely flows
Community's collective mind
Freedom's gentle grasp
